# M2.4 · Numeric & float features

_Curriculum · Domain 0 · ML Foundations · Feature engineering & leakage_

_Save a copy to your Drive_

**Make continuous signals numerically well-behaved without letting validation rows set the ruler.**

We will build a tiny ads-style dataset, apply $\log(1+x)$ to a skewed spend column, and compare a correct train-only scaler with a leaked scaler fitted on all rows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(7)

## Build a skewed numeric feature

Ad spend is nonnegative and heavy-tailed, so a few campaigns can be much larger than the rest. We create train rows first, then validation rows from a shifted distribution to make leakage visible.

In [ ]:
n_train = 120
n_val = 40

train_spend = rng.lognormal(mean=3.0, sigma=1.0, size=n_train)
val_spend = rng.lognormal(mean=4.0, sigma=1.0, size=n_val)

train_clicks = rng.poisson(lam=4.0, size=n_train)
val_clicks = rng.poisson(lam=6.0, size=n_val)

train = pd.DataFrame({"split": "train", "spend": train_spend, "clicks": train_clicks})
val = pd.DataFrame({"split": "val", "spend": val_spend, "clicks": val_clicks})
df = pd.concat([train, val], ignore_index=True)

print(df.groupby("split")["spend"].mean().round(2))

## Compress the heavy tail with log1p

The transform $u=\log(1+x)$ keeps zero valid and turns multiplicative gaps into additive gaps.

In [ ]:
df["log_spend"] = np.log1p(df["spend"])

raw_ratio = df["spend"].quantile(0.95) / df["spend"].quantile(0.50)
log_ratio = df["log_spend"].quantile(0.95) / df["log_spend"].quantile(0.50)

print("raw 95/50 ratio:", round(float(raw_ratio), 2))
print("log 95/50 ratio:", round(float(log_ratio), 2))

## Fit the scaler on train only

The correct standardization is $z=(x-\mu_{\text{train}})/\sigma_{\text{train}}$. Validation rows may be transformed, but they may not help estimate $\mu$ or $\sigma$.

In [ ]:
train_mask = df["split"] == "train"
val_mask = df["split"] == "val"

train_only_scaler = StandardScaler()
train_values = df.loc[train_mask, ["log_spend"]]
train_only_scaler.fit(train_values)

df["z_train_only"] = train_only_scaler.transform(df[["log_spend"]])

print("train mean used:", round(float(train_only_scaler.mean_[0]), 4))
print("train scale used:", round(float(train_only_scaler.scale_[0]), 4))

## Now fit the leaked scaler

This is the subtle bug: fitting on all rows lets validation distribution information move the center and spread.

In [ ]:
leaked_scaler = StandardScaler()
leaked_scaler.fit(df[["log_spend"]])

df["z_leaked"] = leaked_scaler.transform(df[["log_spend"]])

val_mean_train_only = df.loc[val_mask, "z_train_only"].mean()
val_mean_leaked = df.loc[val_mask, "z_leaked"].mean()

print("val mean with train-only scaler:", round(float(val_mean_train_only), 4))
print("val mean with leaked scaler:", round(float(val_mean_leaked), 4))

## Assert the contract

The train-only scaler's mean must equal the training mean, not the all-row mean. The leaked scaler shifts validation values because it uses validation statistics.

In [ ]:
train_log_mean = df.loc[train_mask, "log_spend"].mean()
all_log_mean = df["log_spend"].mean()
mean_shift = abs(train_log_mean - all_log_mean)
val_shift = abs(val_mean_train_only - val_mean_leaked)

assert np.isclose(train_only_scaler.mean_[0], train_log_mean)
assert not np.isclose(train_only_scaler.mean_[0], all_log_mean)
assert val_shift > 0.05
assert mean_shift > 0.05

print("mean shift from using validation rows:", round(float(mean_shift), 4))
print("validation z-mean shift:", round(float(val_shift), 4))

## Visualize the difference

Both curves come from the same validation rows. The only difference is whether validation was allowed to help fit the preprocessing ruler.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

ax.hist(df.loc[val_mask, "z_train_only"], bins=12, alpha=0.65, label="train-only scaler")
ax.hist(df.loc[val_mask, "z_leaked"], bins=12, alpha=0.65, label="leaked scaler")
ax.axvline(0, color="black", linewidth=1)
ax.set_title("validation distribution shifts when the scaler leaks")
ax.set_xlabel("standardized log spend")
ax.set_ylabel("campaigns")
ax.legend()
plt.show()

## Practice

Try each change in the empty cell below.

1. Replace `StandardScaler` with `RobustScaler` and compare validation shifts.
2. Add a missing `spend` value, impute the train median, and create a missingness indicator.
3. Clip `log_spend` at the train 99th percentile before scaling and re-run the assertions.

In [ ]:
# Your turn:
